In [0]:
from pyspark.sql import functions as F

transactions_clean = spark.table("workspace.kkbox.transactions_clean")

transaction_features = (
    transactions_clean
    .groupBy("msno")
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("actual_amount_paid").alias("total_amount_paid"),
        F.avg("actual_amount_paid").alias("avg_amount_paid"),
        F.sum("plan_list_price").alias("total_plan_price"),
        F.avg("payment_plan_days").alias("avg_plan_days"),
        F.sum(
            F.when(F.col("is_auto_renew") == 1, 1).otherwise(0)
        ).alias("auto_renew_count"),
        F.sum(
            F.when(F.col("is_cancel") == 1, 1).otherwise(0)
        ).alias("cancel_count"),
        F.max("transaction_date_clean").alias("latest_transaction_date"),
        F.max("membership_expire_date_clean").alias("latest_expiry_date")
    )
)

display(transaction_features.limit(10))

In [0]:
user_logs_scoped = spark.table("workspace.kkbox.user_logs_scoped")

listening_features = (
    user_logs_scoped
    .groupBy("msno")
    .agg(
        F.countDistinct("log_date").alias("active_days"),
        (
            F.sum("num_25")
            + F.sum("num_50")
            + F.sum("num_75")
            + F.sum("num_985")
            + F.sum("num_100")
        ).alias("total_songs_played"),
        F.sum("num_unq").alias("total_unique_songs"),
        F.sum("total_secs").alias("total_listening_secs"),
        F.avg("total_secs").alias("avg_daily_listening_secs"),
        F.avg(
            F.col("num_25")
            + F.col("num_50")
            + F.col("num_75")
            + F.col("num_985")
            + F.col("num_100")
        ).alias("avg_daily_songs"),
        F.max("total_secs").alias("max_daily_listening_secs")
    )
)

display(listening_features.limit(10))

In [0]:
customer_scope = spark.table("workspace.kkbox.customer_scope")
members_clean = spark.table("workspace.kkbox.members_clean")

customer_features = (
    customer_scope
    .join(
        members_clean,
        on="msno",
        how="left"
    )
    .join(
        transaction_features,
        on="msno",
        how="left"
    )
    .join(
        listening_features,
        on="msno",
        how="left"
    )
)

display(customer_features.limit(20))

In [0]:
from pyspark.sql import functions as F

feature_validation = customer_features.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("msno").alias("unique_customers"),
    F.sum(
        F.when(F.col("msno").isNull(), 1).otherwise(0)
    ).alias("null_customer_ids")
)

display(feature_validation)

In [0]:
customer_features_ml = (
    customer_features
    .fillna(
        0,
        subset=[
            "transaction_count",
            "total_amount_paid",
            "avg_amount_paid",
            "total_plan_price",
            "avg_plan_days",
            "auto_renew_count",
            "cancel_count",
            "active_days",
            "total_songs_played",
            "total_unique_songs",
            "total_listening_secs",
            "avg_daily_listening_secs",
            "avg_daily_songs",
            "max_daily_listening_secs"
        ]
    )
)

display(customer_features_ml.limit(20))

In [0]:
ml_validation = customer_features_ml.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("msno").alias("unique_customers"),
    F.sum(
        F.when(F.col("transaction_count").isNull(), 1).otherwise(0)
    ).alias("null_transaction_count"),
    F.sum(
        F.when(F.col("active_days").isNull(), 1).otherwise(0)
    ).alias("null_active_days"),
    F.sum(
        F.when(F.col("total_listening_secs").isNull(), 1).otherwise(0)
    ).alias("null_listening_secs")
)

display(ml_validation)

In [0]:
customer_features_ml.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.kkbox.customer_features_ml")

In [0]:
churn_validation = (
    customer_features_ml
    .groupBy("is_churn")
    .count()
    .withColumn(
        "churn_rate_pct",
        F.round(F.col("count") / F.lit(200000) * 100, 2)
    )
    .orderBy("is_churn")
)

display(churn_validation)